# Heat Data Download — Quickstart

## What this notebook is for

This notebook is used to **download and prepare Heat hazard input data**
required by the exposure analysis pipeline.

---

## When to use this notebook

Run this notebook:
- before running exposure analysis notebooks,
- or when you need to update heat-related input data.

In most cases, this notebook only needs to be run **once per study area**.

---

## What you need before running

Before running this notebook, make sure that:
- the project environment is correctly installed,
- required data access (e.g. ERA5 / remote datasets) is properly set up.
- OUT_DIR is adapted
- YEARS is adapted
- COUNTRIES is adapted
  


In [ ]:
import os
from pathlib import Path
import cdsapi
import xarray as xr

# -------------------
# CONFIG
# -------------------
YEARS = list(range(2010, 2025))  # 2010..2024

ROOT = Path(".")

# Output directory for final NetCDF files
OUT_DIR = ROOT / "../data/heat"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Country bounding boxes for CDS (ERA5) in order [North, West, South, East]
COUNTRIES = {
    # "TJK": {"area": [41.1, 67.3, 36.5, 75.2]},   # Tajikistan
    #"TKM": {"area": [42.8, 52.2, 35.0, 66.8]},   # Turkmenistan
    # "KGZ": {"area": [43.3, 69.0, 39.0, 80.0]},   # Kyrgyzstan
     "UZB": {"area": [46.0, 55.0, 37.0, 74.0]},   # Uzbekistan
     "KAZ": {"area": [55.5, 46.0, 40.0, 87.5]},   # Kazakhstan
  
}

# -------------------
# MAIN LOOP OVER COUNTRIES
# -------------------
c = cdsapi.Client()

for COUNTRY, cfg in COUNTRIES.items():
    area_bbox = cfg["area"]
    country_lower = COUNTRY.lower()

    print("\n" + "#" * 80)
    print(f"### COUNTRY: {COUNTRY} | AREA = {area_bbox}")
    print("#" * 80)

    # Temp directory for this country
    TMP_DIR = ROOT / f"./_tmp_era5_{country_lower}_heat"
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    # Final NetCDF for this country
    FINAL_NC = OUT_DIR / f"heat_{country_lower}_2010_2024.nc"

    # 1) DOWNLOAD PER YEAR
    hourly_files = []

    for y in YEARS:
        hourly_nc = TMP_DIR / f"era5_t2m_hourly_{country_lower}_{y}.nc"
        if hourly_nc.exists():
            print(f"[skip] already exists: {hourly_nc}")
            hourly_files.append(hourly_nc)
            continue

        print(f"[download] ERA5 hourly t2m for {COUNTRY} {y}")
        c.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": "reanalysis",
                "variable": "2m_temperature",
                "year": str(y),
                "month": [f"{m:02d}" for m in range(1, 13)],
                "day": [f"{d:02d}" for d in range(1, 32)],
                "time": [f"{h:02d}:00" for h in range(24)],
                "area": area_bbox,  # [N, W, S, E]
                "format": "netcdf",
            },
            str(hourly_nc),
        )
        hourly_files.append(hourly_nc)

    # 2) PROCESS PER YEAR -> DAILY TMAX (°C)
    daily_files = []

    for hourly_nc in sorted(hourly_files):
        # parse year from file name: era5_t2m_hourly_<country>_<year>.nc
        y = hourly_nc.stem.split("_")[-1]
        daily_nc = TMP_DIR / f"era5_t2m_dailyTmax_{country_lower}_{y}.nc"

        if daily_nc.exists():
            print(f"[skip] already exists: {daily_nc}")
            daily_files.append(daily_nc)
            continue

        print(f"[process] {hourly_nc.name} -> daily Tmax (°C) for {COUNTRY} {y}")
        ds_hour = xr.open_dataset(hourly_nc, chunks={"time": 240})
        da = ds_hour["t2m"] - 273.15  # Kelvin -> °C
        da.name = "t2m"
        da.attrs["units"] = "degC"

        time_dim = "time" if "time" in da.dims else "valid_time"
        da_daily_tmax = da.resample({time_dim: "1D"}).max(skipna=True)

        da_daily_tmax.to_dataset(name="t2m").to_netcdf(daily_nc)
        daily_files.append(daily_nc)
        ds_hour.close()

    # 3) CONCAT ALL YEARS -> SINGLE DAILY NETCDF FOR THIS COUNTRY
    print(f"[concat] assembling all yearly daily Tmax files for {COUNTRY} -> {FINAL_NC.name}")

    daily_files = sorted(daily_files)
    if not daily_files:
        print(f"[warn] No daily files found for {COUNTRY}, skipping concat.")
        continue

    # Safer multi-file open: no parallel, with chunks, minimal metadata work
    ds_all = xr.open_mfdataset(
        [str(f) for f in daily_files],
        combine="by_coords",
        parallel=False,               # avoid HDF5/dask threading issues
        chunks={"time": 365},         # keep time chunked (~1 year per chunk)
        data_vars="minimal",          # lighter combine
        coords="minimal",
        compat="override",
    )

    ds_all["t2m"].attrs["long_name"] = "Daily maximum 2m air temperature"
    ds_all["t2m"].attrs["units"] = "degC"

    ds_all.to_netcdf(FINAL_NC)
    ds_all.close()

    print(f"Done for {COUNTRY}: {FINAL_NC.resolve()}")
